# Módulo 11: Funciones avanzadas

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m11-funciones-avanzadas.ipynb](m11-funciones-avanzadas.ipynb). No es una tarea entregada del programa.

En el Módulo 05 vimos cómo definir funciones con `def`. En Python, las funciones son además **objetos**: se pueden guardar en variables, pasar como argumento y devolver como resultado. Esa idea es la base de la **programación funcional** y de herramientas que aparecen constantemente en ciencia de datos, como `map`, `filter`, `reduce`, `sorted(key=...)` o `DataFrame.apply`.

1. Funciones como objetos
2. Funciones anónimas con `lambda`
3. `map` y los iteradores perezosos
4. `filter`
5. `functools.reduce`
6. `any` y `all` con expresiones generadoras
7. Comprensiones de listas
8. `*args` y `**kwargs`
9. Un adelanto: decoradores
10. Errores frecuentes
11. Ejercicio

## 1. Funciones como objetos

El nombre de una función es una variable que apunta a un objeto función. Si se escribe sin paréntesis, no se ejecuta: se hace referencia a ella. Eso permite pasarla a otra función, que decidirá cuándo llamarla. A una función que recibe o devuelve funciones se le llama **función de orden superior**.

In [1]:
def al_cuadrado(x):
    return x ** 2

def aplicar_a_todos(funcion, valores):
    """Devuelve una lista con funcion(v) para cada v."""
    resultado = []
    for v in valores:
        resultado.append(funcion(v))
    return resultado

operacion = al_cuadrado            # sin paréntesis: referencia, no llamada
print(operacion(4))
print(aplicar_a_todos(al_cuadrado, [1, 2, 3]))
print(aplicar_a_todos(abs, [-5, 3, -1]))   # también funciones integradas

16
[1, 4, 9]
[5, 3, 1]


Una función también puede **devolver** otra función. La función interna recuerda las variables del entorno donde se creó; a eso se le llama **clausura** (*closure*).

In [2]:
def crear_multiplicador(factor):
    def multiplicar(x):
        return x * factor      # factor queda "recordado"
    return multiplicar

duplicar = crear_multiplicador(2)
con_iva = crear_multiplicador(1.16)
print(duplicar(21))
print(round(con_iva(250), 2))

42
290.0


## 2. Funciones anónimas con `lambda`

`lambda parametros: expresion` crea una función pequeña sin nombre. Equivale a un `def` cuyo cuerpo es solo un `return expresion`. Su uso natural es como argumento de otra función, donde definir una función con nombre sería excesivo.

| Con `def` | Con `lambda` |
| --- | --- |
| `def area(b, h): return b * h / 2` | `lambda b, h: b * h / 2` |

**Cuándo no usar `lambda`.** Si la lógica necesita más de una expresión, condiciones anidadas o se va a reutilizar, un `def` con nombre y docstring es más legible y se depura mejor (los errores muestran el nombre de la función en lugar de `<lambda>`). Asignar una lambda a una variable, como `f = lambda x: ...`, es justo el caso en que conviene un `def`.

In [3]:
# (ciudad, millones de habitantes)
ciudades = [("Lima", 10.9), ("Bogotá", 7.9), ("Santiago", 6.9), ("Quito", 2.8)]

# sorted con key: ordena según lo que devuelva la función para cada elemento
por_poblacion = sorted(ciudades, key=lambda c: c[1])
print(por_poblacion)

por_largo_nombre = sorted(ciudades, key=lambda c: len(c[0]), reverse=True)
print(por_largo_nombre)

[('Quito', 2.8), ('Santiago', 6.9), ('Bogotá', 7.9), ('Lima', 10.9)]
[('Santiago', 6.9), ('Bogotá', 7.9), ('Quito', 2.8), ('Lima', 10.9)]


## 3. `map` y los iteradores perezosos

`map(funcion, iterable)` aplica la función a cada elemento. No devuelve una lista, sino un **iterador perezoso** (*lazy*): no calcula nada hasta que se le piden los valores, y los entrega uno a uno. Por eso, para ver el resultado completo, se envuelve en `list()`.

In [4]:
precios = [120, 80, 45.5]
con_descuento = map(lambda p: p * 0.9, precios)

print(type(con_descuento).__name__)   # un objeto map, todavía sin calcular
print(list(con_descuento))      # list() consume el iterador
print(list(con_descuento))      # ya está agotado: lista vacía

map
[108.0, 72.0, 40.95]
[]


La pereza ahorra memoria cuando hay muchos datos o cuando solo se necesitan los primeros resultados. `map` también acepta varios iterables y avanza por ellos en paralelo:

In [5]:
cantidades = [2, 5, 1]
print(list(map(lambda p, c: p * c, precios, cantidades)))

palabras = "la ciencia de datos empieza con preguntas".split()
print(list(map(str.upper, palabras)))    # un método como función

[240, 400, 45.5]
['LA', 'CIENCIA', 'DE', 'DATOS', 'EMPIEZA', 'CON', 'PREGUNTAS']


## 4. `filter`

`filter(funcion, iterable)` conserva solo los elementos para los que la función devuelve un valor verdadero. Como `map`, devuelve un iterador perezoso. La función que decide se llama **predicado**.

In [6]:
temperaturas = [18.5, 31.2, 25.0, 33.8, 12.1, 29.9]
calurosas = filter(lambda t: t > 30, temperaturas)
print(list(calurosas))

# Con None como función, filter conserva los valores "verdaderos" (descarta 0, "", None...)
respuestas = ["sí", "", "no", None, "tal vez", ""]
print(list(filter(None, respuestas)))

[31.2, 33.8]
['sí', 'no', 'tal vez']


## 5. `functools.reduce`

`reduce(funcion, iterable, inicial)` combina los elementos de dos en dos hasta dejar un solo valor. La función recibe el **acumulado** y el siguiente elemento. Para `[a, b, c, d]` con suma, calcula `((a + b) + c) + d`.

El tercer argumento, el **valor inicial**, es el punto de partida del acumulado. Además, protege contra la lista vacía: sin él, `reduce` no tiene con qué empezar y lanza un `TypeError`.

In [7]:
from functools import reduce

ventas = [250, 400, 175, 90]
total = reduce(lambda acumulado, x: acumulado + x, ventas)
maximo = reduce(lambda a, b: a if a > b else b, ventas)
print(total, maximo)

try:
    reduce(lambda a, b: a + b, [])
except TypeError as e:
    print("TypeError:", e)

print(reduce(lambda a, b: a + b, [], 0))    # con valor inicial: devuelve 0

915 400
TypeError: reduce() of empty iterable with no initial value
0


El valor inicial debe ser el **elemento neutro** de la operación: 0 para la suma, 1 para el producto, `""` para concatenar texto. Para muchas reducciones comunes ya existen funciones integradas más claras: `sum`, `max`, `min` y `math.prod`.

In [8]:
import math

factores = [1.05, 1.02, 0.97]           # crecimiento de tres años
print(round(reduce(lambda a, b: a * b, factores, 1), 4))
print(round(math.prod(factores), 4))    # lo mismo, más legible
print(math.prod([]))                    # el producto vacío es 1

1.0389
1.0389
1


## 6. `any` y `all` con expresiones generadoras

`any(iterable)` devuelve `True` si **algún** elemento es verdadero; `all(iterable)`, si **todos** lo son. Se combinan bien con una **expresión generadora**, `(expresion for x in iterable)`, que produce los valores de uno en uno sin construir una lista. Además se detienen en cuanto conocen la respuesta.

| Llamada | Con iterable vacío |
| --- | --- |
| `any([])` | `False` |
| `all([])` | `True` |

In [9]:
contrasena = "datos2024"

print(any(c.isdigit() for c in contrasena))    # ¿tiene algún dígito?
print(any(c.isupper() for c in contrasena))    # ¿tiene alguna mayúscula?
print(all(c.isalnum() for c in contrasena))    # ¿todo es letra o número?

codigos = ["MX01", "CO22", "pe7", "AR-3"]
validos = list(filter(lambda s: s[:2].isupper() and s[2:].isdigit(), codigos))
print(validos)

True
False
True
['MX01', 'CO22']


## 7. Comprensiones de listas

En Python, la forma más idiomática de transformar y filtrar es la **comprensión de listas**: `[expresion for x in iterable if condicion]`. Equivale a un `map` y un `filter` combinados, y suele leerse mejor que ambos con `lambda`.

| Con funciones | Con comprensión |
| --- | --- |
| `list(map(lambda x: x * 2, datos))` | `[x * 2 for x in datos]` |
| `list(filter(lambda x: x > 0, datos))` | `[x for x in datos if x > 0]` |
| `list(map(lambda x: x * 2, filter(lambda x: x > 0, datos)))` | `[x * 2 for x in datos if x > 0]` |

`map` sigue siendo cómodo cuando la función ya existe con nombre, como en `map(len, palabras)` o `map(float, textos)`.

In [10]:
mediciones = ["12.5", "n/d", "8.1", "", "15.0"]

# Convertir a float solo las mediciones numéricas
con_funciones = list(map(float, filter(lambda m: m.replace(".", "", 1).isdigit(), mediciones)))
con_comprension = [float(m) for m in mediciones if m.replace(".", "", 1).isdigit()]
print(con_funciones)
print(con_comprension)

# Comprensiones de diccionario y conjunto
print({m: len(m) for m in mediciones if m})
print(sorted({len(m) for m in mediciones}))

[12.5, 8.1, 15.0]
[12.5, 8.1, 15.0]
{'12.5': 4, 'n/d': 3, '8.1': 3, '15.0': 4}
[0, 3, 4]


## 8. `*args` y `**kwargs`

Una función puede aceptar un número variable de argumentos:

- `*args` reúne los argumentos **posicionales** sobrantes en una tupla.
- `**kwargs` reúne los argumentos **con nombre** sobrantes en un diccionario.

Los nombres `args` y `kwargs` son una convención; lo que importa son los asteriscos. Al llamar a una función, `*` y `**` hacen lo contrario: desempaquetan una lista o un diccionario en argumentos.

In [11]:
def resumen(*valores, decimales=2, **etiquetas):
    media = sum(valores) / len(valores)
    print(f"n={len(valores)}, media={media:.{decimales}f}", etiquetas)

resumen(3, 4, 8)
resumen(3, 4, 8, decimales=0, unidad="kg", fuente="báscula")

datos = [10, 20, 30, 45]
opciones = {"decimales": 1, "unidad": "cm"}
resumen(*datos, **opciones)    # desempaquetar al llamar

n=3, media=5.00 {}
n=3, media=5 {'unidad': 'kg', 'fuente': 'báscula'}
n=4, media=26.2 {'unidad': 'cm'}


## 9. Un adelanto: decoradores

Un **decorador** es una función que recibe una función y devuelve otra que la envuelve, añadiendo comportamiento antes o después. Combina todo lo anterior: funciones como objetos, clausuras y `*args`/`**kwargs`. La sintaxis `@decorador` sobre un `def` equivale a `funcion = decorador(funcion)`.

In [12]:
def registrar(funcion):
    def envoltura(*args, **kwargs):
        resultado = funcion(*args, **kwargs)
        print(f"{funcion.__name__}{args} -> {resultado}")
        return resultado
    return envoltura

@registrar
def potencia(base, exponente):
    return base ** exponente

potencia(2, 10)
resultado = potencia(3, 2)

potencia(2, 10) -> 1024
potencia(3, 2) -> 9


## 10. Errores frecuentes

**Olvidar `list()` o consumir dos veces un iterador.** `map` y `filter` devuelven iteradores que se agotan tras recorrerse una vez. Si se necesita el resultado varias veces, hay que guardarlo en una lista.

In [13]:
largos = map(len, ["uno", "dos", "tres"])
print("Total:", sum(largos))
print("Máximo:", max(largos, default="iterador agotado"))

largos = list(map(len, ["uno", "dos", "tres"]))
print("Total:", sum(largos), "| Máximo:", max(largos))

Total: 10
Máximo: iterador agotado
Total: 10 | Máximo: 4


**Llamar a la función en lugar de pasarla.** `map(len(), datos)` intenta ejecutar `len` sin argumentos; lo correcto es `map(len, datos)`, sin paréntesis.

**`reduce` sin valor inicial sobre datos que pueden venir vacíos.** Como vimos, lanza `TypeError`. Hay que pasar el elemento neutro de la operación como tercer argumento.

**Probar un filtro solo con datos que pasan.** Si todos los elementos de prueba cumplen la condición, el resultado es igual a la entrada y no se puede saber si el filtro funciona. Un buen ejemplo incluye casos que deben quedar fuera y casos límite.

In [14]:
def tiene_digito(palabra):
    return any(c.isdigit() for c in palabra)

solo_casos_positivos = ["R2D2", "C3PO", "web3"]
print(list(filter(tiene_digito, solo_casos_positivos)))   # igual a la entrada: no prueba nada

prueba_completa = ["R2D2", "robot", "web3", "", "Androide"]
print(list(filter(tiene_digito, prueba_completa)))        # ahora se ve qué descarta

['R2D2', 'C3PO', 'web3']
['R2D2', 'web3']


**Variables de bucle capturadas por una `lambda`.** Una `lambda` creada dentro de un bucle no guarda el valor de la variable en ese momento, sino una referencia a la variable; al final del bucle todas ven el último valor. Se corrige fijando el valor como argumento por defecto.

In [15]:
sumadores = [lambda x: x + i for i in range(3)]
print([f(10) for f in sumadores])          # las tres usan i = 2

sumadores = [lambda x, i=i: x + i for i in range(3)]
print([f(10) for f in sumadores])          # cada una guarda su propio i

[12, 12, 12]
[10, 11, 12]


## 11. Ejercicio

Una tienda en línea registra sus pedidos como tuplas `(cliente, importe, cupon)`:

```python
pedidos = [("ana", 1200.0, "DESC10"), ("luis", 85.5, ""), ("marta", 430.0, "envio2x1"),
           ("pedro", 0.0, ""), ("sofia", 2999.9, "VIP2024"), ("juan", 150.0, "promo")]
```

1. Con una `lambda` y `sorted`, ordena los pedidos de mayor a menor importe.
2. Con `map`, obtén la lista de nombres de cliente con la primera letra en mayúscula.
3. Con `filter`, conserva los pedidos cuyo cupón contiene al menos una mayúscula **y** al menos un dígito.
4. Con `reduce` y un valor inicial adecuado, calcula el importe total de los pedidos con importe mayor que 100. Comprueba que tu expresión devuelve 0 si no hay ninguno.
5. Reescribe los puntos 2 y 3 con comprensiones de listas.
6. Escribe una función `aplicar_descuentos(*importes, **descuentos)` que reste a la suma de importes cada descuento recibido por nombre, y llámala desempaquetando una lista y un diccionario.

### Solución

In [16]:
pedidos = [("ana", 1200.0, "DESC10"), ("luis", 85.5, ""), ("marta", 430.0, "envio2x1"),
           ("pedro", 0.0, ""), ("sofia", 2999.9, "VIP2024"), ("juan", 150.0, "promo")]

# 1. Ordenar por importe, de mayor a menor
print(sorted(pedidos, key=lambda p: p[1], reverse=True)[:3])

# 2. Nombres con mayúscula inicial
print(list(map(lambda p: p[0].capitalize(), pedidos)))

# 3. Cupones con alguna mayúscula y algún dígito
def cupon_valido(pedido):
    cupon = pedido[2]
    return any(c.isupper() for c in cupon) and any(c.isdigit() for c in cupon)

print(list(filter(cupon_valido, pedidos)))

# 4. Total de pedidos mayores que 100, con 0 como valor inicial
grandes = filter(lambda p: p[1] > 100, pedidos)
print(reduce(lambda total, p: total + p[1], grandes, 0))
print(reduce(lambda total, p: total + p[1], filter(lambda p: p[1] > 10_000, pedidos), 0))

[('sofia', 2999.9, 'VIP2024'), ('ana', 1200.0, 'DESC10'), ('marta', 430.0, 'envio2x1')]
['Ana', 'Luis', 'Marta', 'Pedro', 'Sofia', 'Juan']
[('ana', 1200.0, 'DESC10'), ('sofia', 2999.9, 'VIP2024')]
4779.9
0


In [17]:
# 5. Con comprensiones
print([p[0].capitalize() for p in pedidos])
print([p for p in pedidos if cupon_valido(p)])

# 6. *args y **kwargs
def aplicar_descuentos(*importes, **descuentos):
    total = sum(importes)
    for nombre, monto in descuentos.items():
        print(f"  descuento {nombre}: -{monto}")
        total -= monto
    return total

importes = [p[1] for p in pedidos]
print(aplicar_descuentos(*importes, **{"cupon": 120.0, "lealtad": 50.0}))

['Ana', 'Luis', 'Marta', 'Pedro', 'Sofia', 'Juan']
[('ana', 1200.0, 'DESC10'), ('sofia', 2999.9, 'VIP2024')]
  descuento cupon: -120.0
  descuento lealtad: -50.0
4695.4


## Resumen

- Las funciones son objetos: se pasan como argumento, se devuelven y pueden recordar su entorno (clausuras).
- `lambda` sirve para funciones cortas de una expresión usadas como argumento; para algo más, mejor `def`.
- `map` y `filter` devuelven **iteradores perezosos**: hay que convertirlos con `list()` y solo se recorren una vez.
- `reduce` acumula de dos en dos; el valor inicial (el elemento neutro) evita el error con iterables vacíos.
- `any`/`all` con expresiones generadoras y las **comprensiones** son la alternativa idiomática en Python.
- `*args` y `**kwargs` reciben argumentos variables, y son la base de los decoradores.

En el **Módulo 12** se organiza el código en módulos y paquetes, y se lee información desde archivos.